In [1]:
import boto3
import time
bedrock = boto3.client('bedrock', region_name='us-east-2')
response = bedrock.list_foundation_models()
print(f"Available models: {len(response['modelSummaries'])}")
bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-2')

Available models: 96


**Pipeline:** question → model decides (answer or call a tool) → run tool(s) → send results back → repeat until the model gives a final answer. Every decision is logged.

In [2]:
import json
import re
import math
import ast
import operator
import sqlite3
import logging
import urllib.parse
import urllib.request
from datetime import datetime, timezone

MODEL_ID = 'us.anthropic.claude-sonnet-4-6'
bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-2')

In [3]:
# --- Task 4: detailed logging of model decisions ---
# Every step of the agent is recorded twice: in the standard `logging` module (readable console output)
# and in `AgentTrace.events` (structured, can be saved as JSON lines for later analysis).
logger = logging.getLogger("agent")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s | %(message)s", datefmt="%H:%M:%S"))
    logger.addHandler(handler)
    logger.propagate = False


class AgentTrace:
    """Structured record of everything that happened during one agent run."""
    def __init__(self):
        self.events = []

    def log(self, event, **data):
        self.events.append({"t": datetime.now().isoformat(timespec="milliseconds"), "event": event, **data})

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            for e in self.events:
                f.write(json.dumps(e, ensure_ascii=False, default=str) + "\n")

    def summary(self):
        calls = [e for e in self.events if e["event"] == "tool_call"]
        steps = [e for e in self.events if e["event"] == "model_response"]
        return {"model_calls": len(steps),
                "tool_calls": len(calls),
                "tools_used": sorted({c["tool"] for c in calls}),
                "tool_errors": sum(1 for e in self.events if e["event"] == "tool_result" and e["status"] == "error"),
                "input_tokens": sum(s["input_tokens"] for s in steps),
                "output_tokens": sum(s["output_tokens"] for s in steps),
                "model_seconds": round(sum(s["latency"] for s in steps), 2)}

**Tools.** Each tool is a Python function plus a JSON schema (`toolSpec`) that tells the model what it does and which arguments it takes.
- `calculator`: safe arithmetic (parsed with `ast`, no `eval`)
- `web_search`: searches Wikipedia's public API (no key required)
- `query_database`: read-only SQL over a small in-memory SQLite database
- `get_current_time`: current date/time in UTC

In [4]:
# --- Task 2: more tools ---
# 1) Calculator: only numbers, + - * / // % ** and a few math functions are allowed.
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.FloorDiv: operator.floordiv, ast.Mod: operator.mod, ast.Pow: operator.pow,
        ast.USub: operator.neg, ast.UAdd: operator.pos}
_FUNCS = {"sqrt": math.sqrt, "log": math.log, "log10": math.log10, "sin": math.sin, "cos": math.cos,
          "tan": math.tan, "abs": abs, "round": round, "min": min, "max": max}
_CONSTS = {"pi": math.pi, "e": math.e}


def calculator(expression):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.operand))
        if isinstance(node, ast.Name) and node.id in _CONSTS:
            return _CONSTS[node.id]
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in _FUNCS:
            return _FUNCS[node.func.id](*[ev(a) for a in node.args])
        raise ValueError(f"Unsupported expression element: {ast.dump(node)[:60]}")
    return {"expression": expression, "result": ev(ast.parse(expression, mode="eval").body)}


# 2) Web search: Wikipedia search API (returns title + snippet of the top results).
def web_search(query, max_results=3):
    params = urllib.parse.urlencode({"action": "query", "list": "search", "srsearch": query,
                                     "srlimit": max_results, "format": "json"})
    request = urllib.request.Request(f"https://en.wikipedia.org/w/api.php?{params}",
                                     headers={"User-Agent": "bedrock-tool-use-practice/1.0"})
    with urllib.request.urlopen(request, timeout=10) as response:
        hits = json.loads(response.read())["query"]["search"]
    clean = lambda s: re.sub(r"<[^>]+>", "", s).replace("&quot;", '"')
    return {"query": query, "results": [{"title": h["title"], "snippet": clean(h["snippet"])} for h in hits]}


# 3) Database: in-memory SQLite with sample data. Only a single SELECT statement is accepted.
db = sqlite3.connect(":memory:", check_same_thread=False)
db.executescript("""
CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, product TEXT, quantity INTEGER, unit_price REAL, order_date TEXT);
INSERT INTO customers VALUES (1,'Ana Torres','Mexico'),(2,'John Smith','USA'),(3,'Lucia Gomez','Argentina'),(4,'Pierre Dubois','France');
INSERT INTO orders VALUES
 (1,1,'Laptop',2,1200.0,'2025-01-15'),(2,1,'Mouse',5,25.5,'2025-02-03'),
 (3,2,'Monitor',3,310.0,'2025-02-20'),(4,3,'Laptop',1,1150.0,'2025-03-11'),
 (5,3,'Keyboard',4,80.0,'2025-03-12'),(6,4,'Monitor',2,300.0,'2025-04-02'),
 (7,2,'Laptop',1,1250.0,'2025-04-18');
""")


def query_database(sql, max_rows=50):
    statement = sql.strip().rstrip(";")
    if not statement.lower().startswith("select") or ";" in statement:
        raise ValueError("Only a single SELECT statement is allowed.")
    cursor = db.execute(statement)
    columns = [c[0] for c in cursor.description]
    rows = cursor.fetchmany(max_rows)
    return {"columns": columns, "rows": rows, "row_count": len(rows)}


def get_current_time():
    return {"utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")}


# Tool registry: name -> (function, toolSpec). The agent only needs this dict, so adding a tool is one entry.
def _spec(name, description, properties, required=()):
    return {"toolSpec": {"name": name, "description": description,
                         "inputSchema": {"json": {"type": "object", "properties": properties, "required": list(required)}}}}


TOOLS = {
    "calculator": (calculator, _spec(
        "calculator", "Evaluate a math expression exactly. Supports + - * / // % **, parentheses, pi, e, "
                      "sqrt, log, log10, sin, cos, tan, abs, round, min, max. Use it for any non-trivial arithmetic.",
        {"expression": {"type": "string", "description": "e.g. '(1200*2 + 25.5*5) * 1.16'"}}, ["expression"])),
    "web_search": (web_search, _spec(
        "web_search", "Search the web (Wikipedia) for general knowledge, facts, definitions or events.",
        {"query": {"type": "string", "description": "Search terms"},
         "max_results": {"type": "integer", "description": "Number of results (default 3)"}}, ["query"])),
    "query_database": (query_database, _spec(
        "query_database", "Run a read-only SQL SELECT against the sales database. Tables: "
                          "customers(id, name, country) and orders(id, customer_id, product, quantity, unit_price, order_date).",
        {"sql": {"type": "string", "description": "A single SQLite SELECT statement"}}, ["sql"])),
    "get_current_time": (get_current_time, _spec(
        "get_current_time", "Get the current date and time in UTC.", {})),
}
print("Tools:", list(TOOLS))
for expr in ["2**10 + sqrt(144)", "(1200*2 + 25.5*5) * 1.16"]:
    print(expr, "->", calculator(expr)["result"])
print(query_database("SELECT country, COUNT(*) AS orders FROM orders JOIN customers ON customers.id = customer_id GROUP BY country"))

Tools: ['calculator', 'web_search', 'query_database', 'get_current_time']
2**10 + sqrt(144) -> 1036.0
(1200*2 + 25.5*5) * 1.16 -> 2931.8999999999996
{'columns': ['country', 'orders'], 'rows': [('Argentina', 2), ('France', 1), ('Mexico', 2), ('USA', 2)], 'row_count': 4}


**The agent.** `Agent.run()` implements the full tool-calling cycle with `converse`:
1. Send the conversation + tool definitions to the model.
2. If `stopReason == "tool_use"`, run every requested tool (the model may request several at once) and append the results as a `toolResult` user message.
3. Call the model again with the results. Repeat until it answers in plain text (`end_turn`) or `max_iterations` is hit.

The model itself decides *whether* to use a tool, *which* one, and *with what arguments*; the code only executes them. Tool errors are returned to the model (`status: "error"`) so it can fix its call and retry.

In [5]:
# --- Task 1 + 3: complete tool-calling cycle, with an agent that decides which tools to use ---
class Agent:
    def __init__(self, tools=TOOLS, model_id=MODEL_ID, system=None, max_iterations=8, max_tokens=1000, verbose=True):
        self.tools = tools
        self.model_id = model_id
        self.max_iterations = max_iterations
        self.max_tokens = max_tokens
        self.verbose = verbose
        self.system = system or (
            "You are a helpful assistant with access to tools. Decide for yourself whether a tool is needed: "
            "answer directly when you already know the answer, use the calculator for arithmetic, the database for "
            "questions about customers/orders, web_search for general facts. Before calling tools, briefly say what "
            "you are going to do and why. You may chain tools: use the output of one as the input of the next.")
        self.trace = AgentTrace()
        self.messages = []

    def _log(self, event, message, **data):
        self.trace.log(event, **data)
        if self.verbose:
            logger.info(message)

    def _run_tool(self, block):
        name, args = block["name"], block["input"]
        self._log("tool_call", f"  TOOL CALL   {name}({json.dumps(args, ensure_ascii=False)})", tool=name, input=args, id=block["toolUseId"])
        start = datetime.now()
        try:
            if name not in self.tools:
                raise KeyError(f"Unknown tool '{name}'")
            output, status = self.tools[name][0](**args), "success"
        except Exception as e:
            output, status = {"error": f"{type(e).__name__}: {e}"}, "error"
        elapsed = (datetime.now() - start).total_seconds()
        text = json.dumps(output, ensure_ascii=False, default=str)
        self._log("tool_result", f"  TOOL RESULT [{status}, {elapsed:.2f}s] {text[:200]}{'...' if len(text) > 200 else ''}",
                  tool=name, status=status, output=output, seconds=elapsed)
        return {"toolResult": {"toolUseId": block["toolUseId"], "content": [{"text": text}], "status": status}}

    def run(self, question, history=None):
        """Run the agent on `question`. Returns the final answer text."""
        self.trace = AgentTrace()
        messages = list(history or []) + [{"role": "user", "content": [{"text": question}]}]
        self._log("question", f"QUESTION: {question}", question=question)
        tool_config = {"tools": [spec for _, spec in self.tools.values()]}

        for iteration in range(1, self.max_iterations + 1):
            start = datetime.now()
            response = bedrock_runtime.converse(
                modelId=self.model_id, messages=messages, system=[{"text": self.system}],
                toolConfig=tool_config, inferenceConfig={"maxTokens": self.max_tokens, "temperature": 0})
            latency = (datetime.now() - start).total_seconds()
            message, stop_reason, usage = response["output"]["message"], response["stopReason"], response["usage"]
            messages.append(message)

            thoughts = " ".join(b["text"] for b in message["content"] if "text" in b).strip()
            tool_uses = [b["toolUse"] for b in message["content"] if "toolUse" in b]
            self._log("model_response",
                      f"[iter {iteration}] MODEL stop_reason={stop_reason} | {usage['inputTokens']} in / {usage['outputTokens']} out | {latency:.2f}s"
                      + (f"\n  REASONING   {thoughts}" if thoughts and tool_uses else "")
                      + (f"\n  DECISION    call {len(tool_uses)} tool(s): {[t['name'] for t in tool_uses]}" if tool_uses
                         else "\n  DECISION    answer directly (no more tools needed)"),
                      iteration=iteration, stop_reason=stop_reason, text=thoughts, tools_requested=[t["name"] for t in tool_uses],
                      input_tokens=usage["inputTokens"], output_tokens=usage["outputTokens"], latency=latency)

            if stop_reason != "tool_use":
                self._log("final_answer", f"FINAL ANSWER ({iteration} model call(s))", answer=thoughts)
                self.messages = messages
                return thoughts

            results = [self._run_tool(t) for t in tool_uses]
            messages.append({"role": "user", "content": results})

        self._log("max_iterations", f"STOPPED: reached max_iterations={self.max_iterations}")
        self.messages = messages
        return "I could not finish within the allowed number of steps."


agent = Agent()
print(agent.run("What is 15% of 2,340, plus the square root of 2025?"))

01:19:46 | QUESTION: What is 15% of 2,340, plus the square root of 2025?
01:19:49 | [iter 1] MODEL stop_reason=tool_use | 983 in / 101 out | 3.08s
  REASONING   I'll calculate both parts simultaneously using the calculator tool — 15% of 2,340 and the square root of 2025 — then add them together.
  DECISION    call 1 tool(s): ['calculator']
01:19:49 |   TOOL CALL   calculator({"expression": "(0.15 * 2340) + sqrt(2025)"})
01:19:49 |   TOOL RESULT [success, 0.00s] {"expression": "(0.15 * 2340) + sqrt(2025)", "result": 396.0}
01:19:51 | [iter 2] MODEL stop_reason=end_turn | 1122 in / 72 out | 1.52s
  DECISION    answer directly (no more tools needed)
01:19:51 | FINAL ANSWER (2 model call(s))


Here's the breakdown:

- **15% of 2,340** = 0.15 × 2,340 = **351**
- **√2025** = **45**
- **351 + 45 = 396**

The final answer is **396**! 🎯


**Multi-step runs.** The next questions force several iterations: a database lookup followed by a calculation, and independent tools used in the same question. Watch the log: each iteration shows the model's reasoning, its decision, the tool calls and their results.

In [6]:
# Multi-iteration: query the database, then compute with the calculator
print(agent.run("Which customer spent the most in total, and what would their total be with 16% VAT added?"))
print("\nSUMMARY:", agent.trace.summary())

01:19:51 | QUESTION: Which customer spent the most in total, and what would their total be with 16% VAT added?
01:19:53 | [iter 1] MODEL stop_reason=tool_use | 985 in / 148 out | 1.91s
  REASONING   Great question! I'll start by querying the database to find the total spending per customer, then calculate the VAT on top. Let me fetch that data now.
  DECISION    call 1 tool(s): ['query_database']
01:19:53 |   TOOL CALL   query_database({"sql": "SELECT c.id, c.name, SUM(o.quantity * o.unit_price) AS total_spent FROM customers c JOIN orders o ON c.id = o.customer_id GROUP BY c.id, c.name ORDER BY total_spent DESC LIMIT 1"})
01:19:53 |   TOOL RESULT [success, 0.00s] {"columns": ["id", "name", "total_spent"], "rows": [[1, "Ana Torres", 2527.5]], "row_count": 1}
01:19:54 | [iter 2] MODEL stop_reason=tool_use | 1182 in / 100 out | 1.53s
  REASONING   Got it! **Ana Torres** is the top spender with a total of **$2,527.50**. Now let me calculate her total with 16% VAT added.
  DECISION    call 

Here's the full summary:

| Detail | Amount |
|---|---|
| 👑 Top Customer | **Ana Torres** |
| Total Spent (excl. VAT) | **$2,527.50** |
| VAT (16%) | **$404.40** |
| **Total with VAT** | **$2,931.90** |

**Ana Torres** is the highest-spending customer with **$2,527.50** in purchases. With **16% VAT** applied, her total comes to **$2,931.90**.

SUMMARY: {'model_calls': 3, 'tool_calls': 2, 'tools_used': ['calculator', 'query_database'], 'tool_errors': 0, 'input_tokens': 3488, 'output_tokens': 380, 'model_seconds': 5.31}


In [7]:
# The agent chooses different tools for different parts of the question (web_search + database + calculator)
print(agent.run("Who created the Python programming language and in what year? Also, how many Laptop units were sold "
                "in total, and what is that number multiplied by the year Python was first released?"))
print("\nSUMMARY:", agent.trace.summary())

01:19:56 | QUESTION: Who created the Python programming language and in what year? Also, how many Laptop units were sold in total, and what is that number multiplied by the year Python was first released?
01:19:59 | [iter 1] MODEL stop_reason=tool_use | 1001 in / 181 out | 2.49s
  REASONING   Great set of questions! I'll handle all of these simultaneously. I already know Python's creator and release year, but let me verify with a web search, fetch the Laptop sales data from the database, and then I'll use the calculator once I have all the pieces.
  DECISION    call 2 tool(s): ['web_search', 'query_database']
01:19:59 |   TOOL CALL   web_search({"query": "Who created Python programming language and in what year was it first released?"})
01:20:11 |   TOOL RESULT [success, 11.98s] {"query": "Who created Python programming language and in what year was it first released?", "results": [{"title": "Python (programming language)", "snippet": "working on Python in the late 1980s as a...
01:20:

Here's a full summary of the answers:

1. 🐍 **Python's Creator & Release Year:** Python was created by **Guido van Rossum** and was first released in **1991**.

2. 💻 **Total Laptop Units Sold:** A total of **4 Laptop units** were sold.

3. 🔢 **4 (Laptops) × 1991 (Python's release year) = 7,964**

SUMMARY: {'model_calls': 4, 'tool_calls': 4, 'tools_used': ['calculator', 'query_database', 'web_search'], 'tool_errors': 0, 'input_tokens': 5974, 'output_tokens': 488, 'model_seconds': 7.51}


In [8]:
# No tool needed -> the model should answer directly in a single call
print(agent.run("Write a haiku about the ocean."))
print("\nSUMMARY:", agent.trace.summary())

# Guardrail: the write request is rejected by the tool, the error goes back to the model, which explains it
print(agent.run("Delete all rows from the orders table."))
print("\nSUMMARY:", agent.trace.summary())

01:20:27 | QUESTION: Write a haiku about the ocean.
01:20:28 | [iter 1] MODEL stop_reason=end_turn | 971 in / 45 out | 1.49s
  DECISION    answer directly (no more tools needed)
01:20:28 | FINAL ANSWER (1 model call(s))
01:20:28 | QUESTION: Delete all rows from the orders table.


Here's a haiku about the ocean:

**Waves kiss the shoreline,**
**Endless blue meets silver sky,**
**The sea breathes in peace.**

🌊

SUMMARY: {'model_calls': 1, 'tool_calls': 0, 'tools_used': [], 'tool_errors': 0, 'input_tokens': 971, 'output_tokens': 45, 'model_seconds': 1.49}


01:20:31 | [iter 1] MODEL stop_reason=end_turn | 971 in / 218 out | 2.95s
  DECISION    answer directly (no more tools needed)
01:20:31 | FINAL ANSWER (1 model call(s))


I'm sorry, but I **cannot** do that. Here's why:

1. **Read-only access**: My database tool only supports **`SELECT` statements**. I have no ability to execute `DELETE`, `UPDATE`, `INSERT`, or any other data-modifying SQL commands.

2. **Safety by design**: Even if I had write access, deleting all rows from the `orders` table would be a **destructive, irreversible action** that could cause serious data loss. Such operations should only be performed by authorized database administrators with proper backups in place.

If you need to manage your database data, I recommend:
- Contacting your **database administrator**.
- Using a **database management tool** (e.g., pgAdmin, DBeaver, MySQL Workbench) with appropriate credentials and permissions.
- Ensuring you have a **backup** before performing any destructive operations.

Is there anything else I can help you with, such as querying the orders data?

SUMMARY: {'model_calls': 1, 'tool_calls': 0, 'tools_used': [], 'tool_errors': 0, 'input_tok

**Inspecting the decision log.** `agent.trace.events` holds every decision as structured data. It can be filtered, printed as a table, or saved as JSON lines.

In [9]:
agent.run("How many orders did each customer place? Show the customer names.")
for e in agent.trace.events:
    if e["event"] == "model_response":
        print(f"iter {e['iteration']}: stop={e['stop_reason']:<9} tools={e['tools_requested']} tokens={e['input_tokens']}+{e['output_tokens']} latency={e['latency']:.2f}s")
    elif e["event"] == "tool_call":
        print(f"        -> {e['tool']}({e['input']})")
    elif e["event"] == "tool_result":
        print(f"        <- {e['status']} in {e['seconds']:.2f}s")

agent.trace.save("agent_trace.jsonl")
print("\nSaved", len(agent.trace.events), "events to agent_trace.jsonl")

01:20:31 | QUESTION: How many orders did each customer place? Show the customer names.
01:20:33 | [iter 1] MODEL stop_reason=tool_use | 976 in / 134 out | 1.62s
  REASONING   I'll query the database joining the `customers` and `orders` tables to count the number of orders per customer, including their names.
  DECISION    call 1 tool(s): ['query_database']
01:20:33 |   TOOL CALL   query_database({"sql": "\nSELECT c.name, COUNT(o.id) AS order_count\nFROM customers c\nLEFT JOIN orders o ON c.id = o.customer_id\nGROUP BY c.id, c.name\nORDER BY order_count DESC\n"})
01:20:33 |   TOOL RESULT [success, 0.00s] {"columns": ["name", "order_count"], "rows": [["Ana Torres", 2], ["John Smith", 2], ["Lucia Gomez", 2], ["Pierre Dubois", 1]], "row_count": 4}
01:20:35 | [iter 2] MODEL stop_reason=end_turn | 1177 in / 141 out | 1.85s
  DECISION    answer directly (no more tools needed)
01:20:35 | FINAL ANSWER (2 model call(s))


iter 1: stop=tool_use  tools=['query_database'] tokens=976+134 latency=1.62s
        -> query_database({'sql': '\nSELECT c.name, COUNT(o.id) AS order_count\nFROM customers c\nLEFT JOIN orders o ON c.id = o.customer_id\nGROUP BY c.id, c.name\nORDER BY order_count DESC\n'})
        <- success in 0.00s
iter 2: stop=end_turn  tools=[] tokens=1177+141 latency=1.85s

Saved 6 events to agent_trace.jsonl


In [10]:
# Interactive agent (leave the input empty or type /quit to exit)
chat_agent = Agent()
history = []
while True:
    question = input("You: ").strip()
    if question.lower() in ("/quit", "/exit", ""):
        break
    print("\nClaude:", chat_agent.run(question, history), "\n")
    history = chat_agent.messages  # keep the whole tool-use conversation as context for the next question

01:22:29 | QUESTION: Take how many orders did each customer place and multiply each by the year that WW2 ended.
01:22:31 | [iter 1] MODEL stop_reason=tool_use | 984 in / 152 out | 2.06s
  REASONING   Great request! I'll need two things:
1. **Order counts per customer** — from the database.
2. **The year WW2 ended** — I already know this (1945), so no search needed.

Let me fetch the order counts now!
  DECISION    call 1 tool(s): ['query_database']
01:22:31 |   TOOL CALL   query_database({"sql": "SELECT c.name, COUNT(o.id) AS order_count FROM customers c LEFT JOIN orders o ON c.id = o.customer_id GROUP BY c.id, c.name"})
01:22:31 |   TOOL RESULT [success, 0.00s] {"columns": ["name", "order_count"], "rows": [["Ana Torres", 2], ["John Smith", 2], ["Lucia Gomez", 2], ["Pierre Dubois", 1]], "row_count": 4}
01:22:33 | [iter 2] MODEL stop_reason=tool_use | 1203 in / 200 out | 1.93s
  REASONING   Got the order counts! Now I'll multiply each by **1945** (the year WW2 ended) simultaneously.
  D


Claude: Here are the results! Each customer's order count multiplied by **1945** (the year WWII ended):

| Customer | Orders | × 1945 |
|---|---|---|
| Ana Torres | 2 | **3,890** |
| John Smith | 2 | **3,890** |
| Lucia Gomez | 2 | **3,890** |
| Pierre Dubois | 1 | **1,945** | 

